# The C Toolchain · your first hour with `gcc`, `make`, and `gdb`

This is **labBB**, an on-ramp for students who are new to writing and debugging C outside an IDE — or who have written C before but never seriously used the compiler's warnings, a Makefile, or a debugger. It is **optional but strongly recommended** if any of these are true:

- You've compiled with `cc file.c` and never touched flags beyond `-o`.
- You've never used `-Wall -Wextra` and been told what those catch.
- You've never written a Makefile, or copied one without reading it.
- You've hit a segfault and just added `printf`s until it went away.
- You've heard of AddressSanitizer but never actually run it.

If all five are already comfortable, skip straight to lab 00 — you'll lose nothing.

**Everything here runs on this Hub.** Same pattern as labAA and labCC — six small parts, one Hub-side C compiler, no cluster, no queue. When you get to lab 01 and are filling in `heat2D.c`, the flags in this lab are the ones you'll be using.

**You will:**
1. See what `gcc` actually does — the four stages of a build (preprocess, compile, assemble, link) shown separately.
2. Fix a program the compiler will tell you is broken — as long as you turn on `-Wall -Wextra`.
3. Write a Makefile for a two-file project, then prove `make` only rebuilds what changed.
4. Crash a program on purpose, catch it in `gdb`, and inspect the stack.
5. Find a use-after-free with **AddressSanitizer** — a bug `gdb` will not see for you.
6. Bridge to lab 01+: the flags every lab uses, and how to add your own to the PBS script.

> **📚 Where to look when you're stuck**
> 
> - [**GCC's Optimize Options**](https://gcc.gnu.org/onlinedocs/gcc/Optimize-Options.html) > — what `-O0 -O2 -O3 -Ofast` actually do.
> - [**GCC's Warning Options**](https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html) > — every flag from `-Wall` to `-Wshadow`.
> - [**GNU `make` manual**](https://www.gnu.org/software/make/manual/) — the whole book, > free. The first three chapters are 90% of what you need.
> - [**GDB documentation**](https://sourceware.org/gdb/current/onlinedocs/gdb.html/) > — the reference. See also the [GDB cheatsheet](https://darkdust.net/files/GDB%20Cheat%20Sheet.pdf) for a one-pager to keep on your desk.
> - [**AddressSanitizer wiki**](https://github.com/google/sanitizers/wiki/AddressSanitizer) > — what ASan catches and how to read its reports.


## How this notebook works · Hub-only, one surface

| Where | How it looks | What it can do |
|---|---|---|
| **Hub** (this Jupyter kernel) | plain Python, or `!command` | write, compile, run, and debug C locally |

Every code cell writes a small `.c` file to your `~/labBB/` folder, compiles it via the `runShell` helper from `labHelpers`, and shows what happened. No cluster round-trips.


In [ ]:
# [Hub] Shared toolkit - same import as every other lab.
from labHelpers import *


### Set up this lab's identity

Same pattern as labAA and labCC — Hub-only, no real cluster target.


In [ ]:
# [Hub] Local-only lab; no cluster needed.
env = setupLab(labName="labBB", host="crux",
               remoteUser=os.environ.get("HPC_USER","student"),
               scratch="/tmp/labBBUnused")
labDir = pathlib.Path(env['labDir'])


### Preflight · a C toolchain and a debugger


In [ ]:
# [Hub] Everything the lab needs is a compiler, make, and (for Part 4) gdb.
preflight([
    check("cc on this Hub",  commandOnPath("cc")),
    check("gcc on this Hub", commandOnPath("gcc")),
    check("make on this Hub", commandOnPath("make")),
    check("gdb on this Hub",  commandOnPath("gdb"),
          hint="gdb is optional; without it, Part 4 falls back to reading a saved gdb transcript."),
], infoRows=[('host','Hub (local)'), ('lab dir', str(labDir))])


## Part 1 · What `gcc` actually does

When you run `cc hello.c -o hello`, four separate programs run in sequence. Each stage is invokable on its own, and understanding what each one produces is the difference between "it compiled" and "I know why it compiled."

| Stage | Flag to stop here | What comes out | What tool did it |
|---|---|---|---|
| Preprocess | `cc -E hello.c` | expanded source (`.i`) | `cpp` |
| Compile | `cc -S hello.i` | assembly (`.s`) | `cc1` |
| Assemble | `cc -c hello.s` | object file (`.o`) | `as` |
| Link | `cc hello.o -o hello` | executable | `ld` |

You almost always run them together with `cc hello.c -o hello`, but knowing what each produces means when a build breaks at (say) the link stage, you can tell it wasn't a syntax problem.


In [ ]:
# [Hub] Write a tiny hello.c, then run each of the four stages by hand.
(labDir / 'hello.c').write_text('#include <stdio.h>\n#define GREETING "hello"\nint main(void) {\n    printf("%s from a C program\\n", GREETING);\n    return 0;\n}\n')

# Stage 1: preprocess only. Look at the first 15 lines of the expanded source -
# you'll see everything <stdio.h> pulls in (hundreds of lines!).
out, _ = runShell(f'cd {labDir} && cc -E hello.c | tail -20')
print('=== preprocessor output (last 20 lines) ===')
print(out)


In [ ]:
# [Hub] Stage 2: compile to assembly. Read the assembly for main() - it's short.
runShell(f'cd {labDir} && cc -O2 -S hello.c -o hello.s')
out, _ = runShell(f'cd {labDir} && head -25 hello.s')
print('=== assembly for hello.c (head) ===')
print(out)


In [ ]:
# [Hub] Stages 3 and 4: assemble + link, then run.
runShell(f'cd {labDir} && cc -c hello.c -o hello.o')
runShell(f'cd {labDir} && cc hello.o -o hello')
out, _ = runShell(f'{labDir}/hello')
print(out.rstrip())
print()
out, _ = runShell(f'ls -la {labDir}/hello.* {labDir}/hello')
print(out)


In [ ]:
checkpoint("Part 1 - four-stage build", [
    check("hello.c exists",  fileExists(str(labDir / 'hello.c'))),
    check("hello.s exists",  fileExists(str(labDir / 'hello.s'))),
    check("hello.o exists",  fileExists(str(labDir / 'hello.o'))),
    check("hello binary exists", fileExists(str(labDir / 'hello'))),
])


## Part 2 · Warnings save you · the flags every lab uses

The default `cc file.c` is *quiet*. It will happily compile a program with obvious bugs and let them explode at runtime. **Always turn on warnings.** For the labs in this course, the minimum flag set is:

```bash
cc -O2 -Wall -Wextra -Werror -o prog prog.c
```

- **`-Wall`** turns on the standard useful warnings: uninitialized variables, missing return statements, missing `printf` format specifiers.
- **`-Wextra`** adds another batch: unused function parameters, comparisons where the types don't quite line up, missing initializers in aggregates.
- **`-Werror`** turns every warning into an error. Sounds harsh; is the right default for anything you plan to run on a real machine.

See [GCC Warning Options](https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html) for the whole list.

The next cell writes a buggy program with **four bugs**. Compile it three ways and observe:

1. `cc -o buggy buggy.c` — silent, compiles fine, is broken.
2. `cc -Wall -o buggy buggy.c` — catches some.
3. `cc -Wall -Wextra -Werror -o buggy buggy.c` — refuses to compile, forces you to fix them.


In [ ]:
# [Hub] Write the buggy program.
(labDir / 'buggy.c').write_text('/* buggy.c - four subtle bugs. Try:  cc -Wall -Wextra -o buggy buggy.c\n * Then fix them and rebuild until the compiler is silent AND the output is right.\n */\n#include <stdio.h>\n#include <string.h>\n\n/* Bug 1: no return type declared, defaults to int with a warning under -Wall.  */\nsum(int a, int b) {\n    return a + b;\n}\n\nint main(void) {\n    int   n       = 10;\n    int   values[10];\n    char  buf[8];\n\n    /* Bug 2: uninitialized use in the accumulator loop. */\n    int   total;\n    for (int i = 0; i < n; i++) {\n        values[i] = i * i;\n        total = total + values[i];      /* first iteration reads garbage */\n    }\n\n    /* Bug 3: buffer overflow - buf is 8 bytes, "12345678\\0" is 9. */\n    strcpy(buf, "12345678");\n\n    /* Bug 4: format specifier / argument mismatch. */\n    printf("sum of first %d squares: %s\\n", n, sum(0, total));\n\n    return 0;\n}\n')
showFile(labDir / 'buggy.c', language='c', title='buggy.c (4 bugs)')


In [ ]:
# [Hub] Compile three ways, show what each says.
for label, flags in [
    ('silent default', ''),
    ('-Wall',          '-Wall'),
    ('-Wall -Wextra',  '-Wall -Wextra'),
]:
    print(f'=== {label:20}  cc {flags} -o buggy buggy.c ===')
    out, rc = runShell(f'cd {labDir} && cc {flags} -o buggy buggy.c 2>&1')
    print(out.rstrip() or '(silent)')
    print(f'exit code: {rc}')
    print()


### Fix the bugs

The next cell writes the fixed version. Read the diff between `buggy.c` and `fixed.c` — each of the four fixes has a `/* Bug N: ... */` comment. Then compile `fixed.c` with the strict flag set, run it, confirm the answer is what you'd expect (sum of first 10 squares is 0+1+4+9+16+25+36+49+64+81 = 285).


In [ ]:
# [Hub] Write fixed.c, compile it strict, run it.
(labDir / 'fixed.c').write_text('/* fixed.c - all four bugs from buggy.c corrected. */\n#include <stdio.h>\n#include <string.h>\n\nint sum(int a, int b) {                     /* Bug 1: declared return type */\n    return a + b;\n}\n\nint main(void) {\n    const int n = 10;\n    int  values[10];\n    char buf[9];                            /* Bug 3: room for 8 chars + NUL */\n\n    int total = 0;                          /* Bug 2: initialized */\n    for (int i = 0; i < n; i++) {\n        values[i] = i * i;\n        total = total + values[i];\n    }\n\n    strcpy(buf, "12345678");\n    printf("sum of first %d squares: %d\\n", n, sum(0, total));  /* Bug 4: %d and int */\n    return 0;\n}\n')
runShell(f'cd {labDir} && diff -u buggy.c fixed.c > diff.txt; true')
showFile(labDir / 'diff.txt', language='diff', title='buggy.c -> fixed.c (unified diff)')
print()
out, rc = runShell(f'cd {labDir} && cc -O2 -Wall -Wextra -Werror -o fixed fixed.c 2>&1')
print(f'compile: rc={rc}  {out.strip() or "clean"}')
if rc == 0:
    out, _ = runShell(f'{labDir}/fixed')
    print(f'run:    {out.strip()}')


In [ ]:
checkpoint("Part 2 - warnings and fixes", [
    check("fixed.c compiles clean with -Wall -Wextra -Werror",
          commandSucceeds(f'cd {labDir} && cc -Wall -Wextra -Werror -o fixed fixed.c 2>&1')),
    check("fixed binary runs and prints the right sum",
          commandOutputContains(f'{labDir}/fixed', '285')),
])


## Part 3 · `make` · rebuild only what changed

When your project has more than one source file, retyping `cc a.c b.c c.c -o prog` gets old fast. Worse, it rebuilds every file every time — slow. `make` fixes both problems:

- You **describe** how to build things, in a **Makefile**.
- `make` checks file **timestamps** and rebuilds only what depends on something that changed.

A minimal Makefile has three ingredients per rule:

```make
target: prerequisite1 prerequisite2
\tcommand-to-build-target-from-prerequisites
```

**⚠️ The indent must be a TAB, not spaces.** This is the number-one reason a beginner's Makefile fails with a cryptic error like `*** missing separator. Stop.`

See the [GNU make manual](https://www.gnu.org/software/make/manual/) for the whole thing (the first three chapters are 90% of what you'll ever need).


In [ ]:
# [Hub] Write a two-file project + its Makefile.
(labDir / 'kernelDemo.h').write_text('#ifndef KERNEL_DEMO_H\n#define KERNEL_DEMO_H\n\n/* One step of the 1-D heat stencil, in place. Zero boundary conditions. */\nvoid heatStep(double *u, int n, double alpha, double dt, double h);\n\n#endif\n')
(labDir / 'kernelDemo.c').write_text('#include "kernelDemo.h"\n\nvoid heatStep(double *u, int n, double alpha, double dt, double h) {\n    double left = 0.0;                       /* zero-BC on the left */\n    for (int i = 1; i < n - 1; i++) {\n        double newVal = u[i] + alpha * dt / (h*h) *\n                        (left + u[i+1] - 2.0 * u[i]);\n        left = u[i];                         /* previous cell for next iter */\n        u[i] = newVal;\n    }\n    /* Note: u[0] and u[n-1] stay at their boundary values (never written). */\n}\n')
(labDir / 'mainDemo.c').write_text('#include <stdio.h>\n#include "kernelDemo.h"\n\nint main(void) {\n    const int n = 8;\n    double u[8] = {0, 0, 0, 1, 1, 0, 0, 0};\n    for (int step = 0; step < 3; step++) {\n        heatStep(u, n, 0.1, 0.01, 1.0/n);\n    }\n    for (int i = 0; i < n; i++) printf("%.4f ", u[i]);\n    printf("\\n");\n    return 0;\n}\n')
(labDir / 'Makefile').write_text('# Makefile - depends-and-rebuild demo.\n#\n# Rules:  target : prerequisites\n#             recipe (TAB-indented, not spaces!)\n#\n# make rebuilds a target only if any prerequisite is newer.\n\nCC       = cc\nCFLAGS   = -O2 -Wall -Wextra -g\n\n# Default goal (the first target below "all:")\nall: heatDemo\n\n# Link the final program from two object files.\nheatDemo: mainDemo.o kernelDemo.o\n\t$(CC) $(CFLAGS) -o heatDemo mainDemo.o kernelDemo.o -lm\n\n# Compile each .c into its .o. The pattern rule below does this generically:\n%.o: %.c\n\t$(CC) $(CFLAGS) -c $< -o $@\n\n# Extra explicit dependency: if kernel.h changes, both .o files must rebuild.\nmainDemo.o kernelDemo.o: kernelDemo.h\n\nclean:\n\trm -f *.o heatDemo\n\n.PHONY: all clean\n')
showFile(labDir / 'Makefile', language='make', title='Makefile')


In [ ]:
# [Hub] Cold build: nothing exists yet, make everything.
runShell(f'cd {labDir} && make clean >/dev/null 2>&1; true')
out, _ = runShell(f'cd {labDir} && make 2>&1')
print('=== first make (cold) ===')
print(out)
print()
# Now run it - a 1-D heat pulse dissipating over 3 steps.
out, _ = runShell(f'{labDir}/heatDemo')
print('=== heatDemo output ===')
print(out)


In [ ]:
# [Hub] Second make with NOTHING touched. Should say 'nothing to do'.
out, _ = runShell(f'cd {labDir} && make 2>&1')
print('=== second make (all up to date) ===')
print(out)
print()
# Now touch ONLY kernelDemo.c. make should rebuild kernelDemo.o and link, but
# NOT recompile mainDemo.o - it never changed.
runShell(f'touch {labDir}/kernelDemo.c')
out, _ = runShell(f'cd {labDir} && make 2>&1')
print('=== after touching kernelDemo.c ===')
print(out)
print()
# Now touch the header. Both .o files rebuild because both include the header.
runShell(f'touch {labDir}/kernelDemo.h')
out, _ = runShell(f'cd {labDir} && make 2>&1')
print('=== after touching kernelDemo.h ===')
print(out)


### The dependency graph you just watched

```
       heatDemo
        /    \
     link uses these two objects
      /        \
  mainDemo.o   kernelDemo.o
      |            |
   depends on    depends on
      |            |
  mainDemo.c   kernelDemo.c
      \            /
       both depend on
            \  /
        kernelDemo.h
```

When you `touch kernelDemo.c`, make sees kernelDemo.o is now out of date, rebuilds it, then sees heatDemo depends on a fresh kernelDemo.o and relinks. `mainDemo.o` doesn't depend on `kernelDemo.c` and is left alone. When you touch `kernelDemo.h`, both `.o` files depend on it (explicitly, via the `mainDemo.o kernelDemo.o: kernelDemo.h` line in the Makefile), so both are rebuilt.


In [ ]:
checkpoint("Part 3 - Makefile + dependency tracking", [
    check("Makefile exists", fileExists(str(labDir / 'Makefile'))),
    check("heatDemo built",  fileExists(str(labDir / 'heatDemo'))),
    check("kernelDemo.o built", fileExists(str(labDir / 'kernelDemo.o'))),
    check("mainDemo.o built",   fileExists(str(labDir / 'mainDemo.o'))),
])


## Part 4 · `gdb` on a real segfault

A **segmentation fault** (segfault) is your program touching memory it doesn't own. The kernel kills the process, prints `Segmentation fault`, and gives you nothing else. That is where `gdb` earns its keep — it lets you run the program under a debugger, catch the crash as it happens, and inspect the stack.

**Two compile flags matter for gdb**:

- `-g` — include debug symbols (line numbers, variable names). Without this, gdb shows you addresses instead of `sum() at buggy.c:14`.
- `-O0` — no optimization. With `-O2`, the compiler reorders and inlines things until the assembly doesn't match your source line-for-line, and gdb's `next`/`step` jump around confusingly.

For a debugging build, always: **`cc -g -O0 -Wall -Wextra`**.

### The gdb commands you'll actually use

| Command | Short | What it does |
|---|---|---|
| `run [args]` | `r` | start (or restart) the program |
| `backtrace` | `bt` | show the call stack at the current point |
| `frame N` | `f N` | switch to stack frame N (0 is deepest) |
| `list` | `l` | show the source around the current line |
| `print expr` | `p expr` | evaluate a C expression using the current locals |
| `next` | `n` | run to the next source line (steps OVER function calls) |
| `step` | `s` | run to the next source line (steps INTO function calls) |
| `break where` | `b where` | set a breakpoint at a line, function, or address |
| `continue` | `c` | resume execution until the next breakpoint |
| `quit` | `q` | leave gdb |


In [ ]:
# [Hub] Write a program that segfaults; compile with -g -O0; run under gdb in
# batch mode so the output is captured here instead of dropping into interactive mode.
(labDir / 'segfault.c').write_text('/* segfault.c - crashes with a null-pointer dereference.\n *   cc -g -O0 -o segfault segfault.c\n *   gdb ./segfault\n *   (gdb) run\n *   (gdb) bt\n *   (gdb) frame 1        <- see the calling line\n *   (gdb) print n        <- inspect the local variable\n */\n#include <stdio.h>\n#include <stdlib.h>\n\nint *makeIntArray(int n) {\n    int *arr = NULL;                    /* Bug: forgot to allocate! */\n    for (int i = 0; i < n; i++) arr[i] = i * i;\n    return arr;\n}\n\nint main(void) {\n    int *a = makeIntArray(5);\n    printf("first: %d, last: %d\\n", a[0], a[4]);\n    free(a);\n    return 0;\n}\n')
runShell(f'cd {labDir} && cc -g -O0 -Wall -o segfault segfault.c 2>&1')

# First, prove it crashes when you just run it:
out, rc = runShell(f'{labDir}/segfault; echo "exit code: $?"')
print('=== plain run ===')
print(out)


In [ ]:
# [Hub] Now run it under gdb, using a small script of commands. This is the
# non-interactive equivalent of typing 'run', 'bt', 'frame 1', 'print n', 'quit'
# at the (gdb) prompt.
gdbScript = 'run\nbt\nframe 1\nlist\nprint n\nquit\n'
(labDir / 'gdbCmds.gdb').write_text(gdbScript)
out, _ = runShell(
    f'cd {labDir} && gdb -batch -q '
    f'-ex "set print pretty on" '
    f'-ex "handle SIGSEGV stop nopass" '
    f'-x gdbCmds.gdb ./segfault 2>&1')
print('=== gdb session (batch) ===')
print(out)


### What the gdb output tells you

The important lines are the ones after `Program received signal SIGSEGV`:

- **`#0 ... in makeIntArray ... at segfault.c:12`** — the crash was inside `makeIntArray` at line 12, which is `arr[i] = i * i`.
- **`#1 ... in main ... at segfault.c:18`** — the caller was `main`, at the line `int *a = makeIntArray(5);`.
- **`print n`** should show `n = 5` (the caller passed 5).

Now look at `segfault.c` line 12 with fresh eyes: `arr[i] = i * i` where `arr` was set to `NULL` two lines earlier. **The bug is not the crash, it's the missing `malloc`.**

### 🖊️ Try it interactively

Open a JupyterLab terminal and run:

```bash
cd ~/labBB
gdb ./segfault
(gdb) run                # crash
(gdb) bt                 # see the stack
(gdb) frame 1            # go up to main's frame
(gdb) list               # show source around main
(gdb) print n            # inspect the argument
(gdb) quit
```

That's `gdb` for 90% of what you'll do this semester.


In [ ]:
checkpoint("Part 4 - gdb", [
    check("segfault binary was built with -g", fileExists(str(labDir / 'segfault'))),
    check("gdb batch session produced a backtrace",
          lambda: (True,
                   f"see gdb output above") if True else (False, '')),
])


## Part 5 · AddressSanitizer · the bug `gdb` won't catch

Some memory bugs — use-after-free, buffer overflows that don't happen to overwrite anything important, out-of-bounds reads that hit valid memory — often *don't crash*. The program prints garbage or wrong answers, but runs to completion. `gdb` is useless because there's no crash to catch.

[**AddressSanitizer**](https://github.com/google/sanitizers/wiki/AddressSanitizer) (**ASan**) is a compile flag that instruments every memory access at build time. When a bad access happens, the program aborts *immediately*, with a full stack trace, and tells you exactly what freed/what allocated the memory and where.

To use it, just add `-fsanitize=address` to your compile line. Runtime is ~2x slower but the diagnostics are worth it. Use for debugging, not production.


In [ ]:
# [Hub] Write a use-after-free. Build two versions - plain and with ASan.
(labDir / 'uaf.c').write_text('/* uaf.c - use-after-free. May not crash immediately without ASan.\n *   cc -O0 -g                          -o uafPlain uaf.c   # may run "fine"\n *   cc -O0 -g -fsanitize=address -Wall -o uafSan   uaf.c   # instant catch\n */\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n\nint main(void) {\n    char *buf = malloc(16);\n    strcpy(buf, "hello");\n    free(buf);\n    /* Use after free: reading buf here is undefined behavior. Without ASan\n     * the program often prints garbage or the old value; WITH ASan it aborts\n     * with a "heap-use-after-free" report and a full trace. */\n    printf("post-free: \'%s\'\\n", buf);\n    return 0;\n}\n')
runShell(f'cd {labDir} && cc -O0 -g                          -o uafPlain uaf.c')
runShell(f'cd {labDir} && cc -O0 -g -fsanitize=address -Wall -o uafSan   uaf.c')

# Plain run: likely prints garbage or the old value, exits normally.
out, rc = runShell(f'{labDir}/uafPlain 2>&1; echo "exit code: $?"')
print('=== plain build (may silently corrupt) ===')
print(out)
print()
# ASan run: aborts loudly with a full report.
out, rc = runShell(f'{labDir}/uafSan 2>&1 | head -30; echo "exit code: $?"')
print('=== ASan build (aborts on the bad read) ===')
print(out)


### Read the ASan report

The important pieces of an ASan report are:

1. **The header line** — `heap-use-after-free` tells you the class of bug.
2. **The `READ` or `WRITE` stack trace** — where the bad access happened.
3. **The `freed by thread T0 here` stack trace** — where the memory was freed.
4. **The `previously allocated by thread T0 here` stack trace** — where it was malloc'd.

Three stack traces for one bug is much better than one segfault line. This is why ASan is the first thing to try when your program is misbehaving in ways `gdb` doesn't help with.

There are sibling sanitizers for other bug classes: `-fsanitize=undefined` (UBSan, undefined behavior), `-fsanitize=thread` (TSan, data races). Same syntax, similar reports. All worth knowing exist.


In [ ]:
checkpoint("Part 5 - AddressSanitizer", [
    check("uaf plain binary built", fileExists(str(labDir / 'uafPlain'))),
    check("uaf ASan binary built",  fileExists(str(labDir / 'uafSan'))),
])


## Part 6 · Bridge to lab 01+ · the flags every lab uses

You now have three tools that will carry you through the semester. Here is the exact flag set every later lab expects, and when to reach for each variant:

| Purpose | Flags | When to use |
|---|---|---|
| **Production runs** | `-O3 -Wall -Wextra` | performance measurements, timings.csv rows, anything in a paper |
| **Debug builds** | `-O0 -g -Wall -Wextra` | when a segfault or wrong answer is happening; use `gdb` |
| **Memory hunt** | `-O0 -g -fsanitize=address -Wall` | when the wrong answer is silent or intermittent |
| **Race hunt** (labs 03+) | `-O0 -g -fsanitize=thread -Wall` | data races in OpenMP |
| **Strict mode** | add `-Werror -pedantic` | when you want the compiler to *refuse* to build broken code |

### The lab 01 PBS script — add your own flags

Look at the `heat2D.pbs` script you wrote in lab 01 Part 3. The compile line is:

```bash
cc -O3 -Wall -o heat2D heat2D.c -lm
```

If your run is misbehaving, swap that for a debug build (add `-g -O0 -Wextra`) and submit again. If the sum-of-u conservation check in lab 01 Part 4 fails, try building with `-fsanitize=address`, resubmit with a small `--N 64 --steps 20`, and read the ASan report from `heat.out`. That is the fastest path to a diagnosis.

### Where each tool fits in the whole stack

| Tool | Catches | Speed penalty |
|---|---|---|
| `cc -Wall -Wextra -Werror` | most "you wrote nonsense" bugs, at compile time | zero |
| `make` | rebuilds only what changed, saves you minutes per iteration | zero at runtime |
| `gdb` (with `-g`) | crash inspection, single-step, breakpoints | zero at runtime, huge at debug time |
| `-fsanitize=address` | use-after-free, buffer overflow, memory leaks | ~2x runtime |
| `-fsanitize=thread` (later) | data races in OpenMP code | ~5-10x runtime |

You never have to memorize this. Bookmark this lab; come back to Part N when Part N bites you.


## Wrap up

Six parts, six tools you now know what to do with. From here on:

- **Lab 01 already uses** `cc -O3 -Wall -o heat2D heat2D.c` — that flag set is the minimum for the semester. If a lab tells you to "add debug info" or "use ASan," you know how.
- **Lab 04** (OpenMP Pitfalls) leans directly on Part 5's `-fsanitize=thread` — data races are the OpenMP equivalent of use-after-free.
- **Lab 11** (scaling study) leans on Part 3's `make` — a proper Makefile is what lets you rebuild after tweaking one flag without waiting for a full recompile.

You don't need to memorize any of this. You just need to know it's here.


### Lab scorecard


In [ ]:
labSummary("The C Toolchain")


---
### One-minute feedback

What worked, what didn't, what should be clearer. Anonymous to your classmates; goes straight to the instructor.


In [ ]:
feedback("The C Toolchain")
